# 🧠 Mental Stress Level Detection — LSTM Pipeline
**Dataset:** WESAD (Wearable Stress and Affect Detection)  
**Model:** Long Short-Term Memory (LSTM)  
**Validation:** Leave-One-Subject-Out (LOSO) Cross-Validation  
**Reference:** Malviya et al. — *Mental Stress Level Detection using LSTM for WESAD Dataset*

---
### 📋 Pipeline Overview
```
Raw Compressed Files → On-The-Fly Load → StandardScaler → LSTM Model → LOSO Metrics
```
### 💾 Memory Strategy
- Files loaded **one subject at a time** — never all subjects in RAM simultaneously
- Explicit `del` + `gc.collect()` + `tf.keras.backend.clear_session()` after every fold
- Mixed precision (float16) to halve GPU VRAM usage
- Small LSTM hidden size (64 units) — expressive but lean
- Gradient clipping to stabilise training on noisy biosignals

## Cell 1 — Environment Setup & Path Configuration

In [ ]:
import os
from google.colab import drive

# ==========================================
# GLOBAL CONFIGURATION  ← Modify here only
# ==========================================
BASE_DIR     = '/content/drive/MyDrive/'
MODEL_NAME   = 'chest_all'   # Keep same as CNN run for apples-to-apples comparison

# Signal keys to use — must match keys inside your compressed pickle windows
SIGNAL_KEYS  = ['chest_ECG', 'chest_EDA', 'chest_EMG', 'chest_RESP', 'chest_TEMP', 'chest_ACC']

# ==========================================
# DYNAMIC PATH GENERATION
# ==========================================
COMPRESSED_DATA_DIR = os.path.join(BASE_DIR, 'compressed')
OUTPUT_DIR          = os.path.join(BASE_DIR, 'DL', 'LSTM', MODEL_NAME)  # Separate from CNN outputs

try:
    print("--> Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)

    if not os.path.exists(COMPRESSED_DATA_DIR):
        raise FileNotFoundError(
            f"[CRITICAL PATH ERROR]\n"
            f"Data directory not found: {COMPRESSED_DATA_DIR}\n"
            f"Double-check BASE_DIR or folder spelling in Drive."
        )

    os.makedirs(os.path.join(OUTPUT_DIR, 'result'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'models'), exist_ok=True)

    print("\n[SUCCESS] Environment verified.")
    print(f"Reading files from : {COMPRESSED_DATA_DIR}")
    print(f"Saving outputs to  : {OUTPUT_DIR}")

except Exception as e:
    print("\n" + "!"*50 + "\n[ERROR DURING SETUP]\n" + "!"*50)
    print(str(e))
    print("!"*50)

## Cell 2 — Mixed Precision & GPU Configuration

In [ ]:
import tensorflow as tf

# ── Mixed precision: compute in float16, accumulate weights in float32
# This halves GPU memory footprint with zero accuracy loss on T4/V100
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

# ── Limit GPU memory growth so Colab RAM is not pre-allocated all at once
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"[GPU] Found {len(gpus)} GPU(s). Memory growth enabled.")
else:
    print("[CPU] No GPU detected. Training will be slower but functional.")

print(f"TensorFlow version  : {tf.__version__}")
print(f"Compute policy      : {mixed_precision.global_policy().name}")

## Cell 3 — Data Loading Utilities (Identical to CNN notebook)

In [ ]:
import gzip
import pickle
import glob
import numpy as np


def load_subject_data(file_path, feature_keys):
    """Loads a single subject file with comprehensive structure validation."""
    subject_id = os.path.basename(file_path).split('_')[0]

    try:
        with gzip.open(file_path, 'rb') as f:
            windows_list = pickle.load(f)
    except Exception as e:
        raise IOError(
            f"Failed to read/decompress pickle for {subject_id}.\nDetails: {e}"
        )

    X_list, y_list = [], []

    for idx, window in enumerate(windows_list):
        window_features = []

        # --- 1. VALIDATE FEATURE KEYS ---
        for key in feature_keys:
            if key not in window:
                raise KeyError(
                    f"\n[DATA STRUCTURE ERROR] Subject: {subject_id} | Window: {idx}\n"
                    f"Missing key: '{key}'\n"
                    f"Available keys: {list(window.keys())}"
                )
            signal = np.array(window[key], dtype=np.float32)
            if signal.ndim == 1:
                signal = np.expand_dims(signal, axis=-1)
            window_features.append(signal)

        # --- 2. VALIDATE TEMPORAL SHAPE ALIGNMENT ---
        try:
            stacked = np.concatenate(window_features, axis=-1)
        except ValueError as shape_err:
            shapes_report = {k: np.array(window[k]).shape for k in feature_keys}
            raise ValueError(
                f"\n[SHAPE MISMATCH] Subject: {subject_id} | Window: {idx}\n"
                f"Shapes: {shapes_report}\nDetails: {shape_err}"
            )

        # --- 3. FIND LABEL KEY WITH FALLBACKS ---
        label_key = None
        for possible in ['label', 'labels', 'Label', 'labels_id', 'final_label']:
            if possible in window:
                label_key = possible
                break

        if label_key is None:
            raise KeyError(
                f"\n[LABEL MISSING] Subject: {subject_id} | Window: {idx}\n"
                f"Available keys: {list(window.keys())}"
            )

        X_list.append(stacked)
        y_list.append(window[label_key])

    return np.array(X_list, dtype=np.float32), np.array(y_list)


print("[OK] Data loading utilities defined.")

## Cell 4 — LSTM Model Architecture

**Design decisions (all memory-motivated):**
| Choice | Value | Why |
|---|---|---|
| LSTM units | 64 | Adequate capacity; doubling costs 4× VRAM |
| Stacked layers | 2 | Adds depth without exploding parameter count |
| `return_sequences` | True → False | First layer passes sequences; second collapses to vector |
| Dropout | 0.4 | Regularise; biosignals are noisy |
| Dense head | 32 → num_classes | Lightweight classifier on top |
| Gradient clip | 1.0 | Prevents LSTM exploding gradients on long sequences |
| Output dtype | float32 | Required when mixed precision is active |

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam


def build_lstm_model(input_shape, num_classes):
    """
    Two-layer stacked LSTM for temporal biosignal classification.

    Args:
        input_shape : tuple (timesteps, num_channels)
        num_classes : int — number of stress/affect categories

    Returns:
        Compiled Keras model
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # ── Layer 1: LSTM — returns full sequence for stacking
        # use_bias=True kept (LSTM relies on biases for forget gate initialisation)
        layers.LSTM(
            units=64,
            return_sequences=True,    # Pass sequence to next LSTM layer
            dropout=0.2,              # Input dropout — applied during training
            recurrent_dropout=0.0,    # Keep at 0: recurrent dropout disables cuDNN kernel → very slow
            kernel_initializer='glorot_uniform',
            recurrent_initializer='orthogonal',
            bias_initializer='zeros',
            unit_forget_bias=True,    # Initialise forget gate bias to 1 (helps long-term memory)
            name='lstm_1'
        ),

        # ── Layer 2: LSTM — collapses sequence to single context vector
        layers.LSTM(
            units=64,
            return_sequences=False,   # Output: (batch, 64) — no more temporal dimension
            dropout=0.2,
            recurrent_dropout=0.0,
            unit_forget_bias=True,
            name='lstm_2'
        ),

        # ── Regularisation
        layers.Dropout(0.4, name='dropout'),

        # ── Dense classification head
        layers.Dense(32, use_bias=False, name='dense_hidden'),
        layers.BatchNormalization(name='bn'),
        layers.Activation('relu', name='relu'),

        # ── Output layer — dtype=float32 is required with mixed precision
        layers.Dense(num_classes, activation='softmax', dtype='float32', name='output')
    ])

    # Adam with gradient clipping (essential for LSTM stability)
    optimizer = Adam(
        learning_rate=1e-3,
        clipnorm=1.0    # Clip gradient norm — prevents exploding gradients
    )

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# ── Quick sanity-check: print model summary with a dummy input_shape
# (Replace (700, 6) with your actual timesteps × channels if you want to verify)
dummy_model = build_lstm_model(input_shape=(700, 6), num_classes=4)
dummy_model.summary()

# Free the dummy model immediately
import gc
del dummy_model
tf.keras.backend.clear_session()
gc.collect()
print("\n[OK] Dummy model freed. Real models will be built fold-by-fold.")

## Cell 5 — Memory-Efficient LOSO Cross-Validation

**Key memory optimisations over a naïve implementation:**
1. Files loaded on-the-fly — never all subjects in RAM at once
2. Training lists deleted immediately after `np.concatenate`
3. Model, arrays and scaler deleted after every fold
4. `tf.keras.backend.clear_session()` clears the TF graph and GPU allocations
5. `gc.collect()` forces Python to release unreferenced memory

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import pandas as pd

# ── Callbacks reused each fold (stateless — safe to share)
def make_callbacks():
    """Returns a list of training callbacks for each fold."""
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,              # Stop if val_loss doesn't improve for 3 epochs
        restore_best_weights=True
    )
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,              # Halve LR on plateau
        patience=2,
        min_lr=1e-6,
        verbose=0
    )
    return [early_stop, reduce_lr]


def run_lstm_loso(data_dir, feature_keys, output_dir, epochs=15, batch_size=32):
    """
    Leave-One-Subject-Out cross-validation with the LSTM model.
    Loads each subject file on-the-fly to keep RAM usage bounded.

    Args:
        data_dir     : path to compressed subject files
        feature_keys : list of signal keys to extract from each window
        output_dir   : where to save CSV results and model
        epochs       : max training epochs per fold (early stopping cuts this short)
        batch_size   : samples per gradient step
    """
    all_files = sorted(glob.glob(os.path.join(data_dir, 'S*_windows_compressed.pkl.gz')))
    if not all_files:
        raise FileNotFoundError(f"No files matching 'S*_windows_compressed.pkl.gz' in {data_dir}")

    subjects = [os.path.basename(f).split('_')[0] for f in all_files]
    loso_results = []

    # ── Peek at first file to infer shape and class count
    print("--> Peeking at first subject to configure LSTM architecture...")
    X_peek, y_peek = load_subject_data(all_files[0], feature_keys)
    _, num_timesteps, num_channels = X_peek.shape
    num_classes = int(y_peek.max() + 1)
    del X_peek, y_peek
    gc.collect()

    print(f"--> LSTM Input shape per window : ({num_timesteps}, {num_channels})")
    print(f"--> Number of output classes    : {num_classes}")
    print(f"--> Total LOSO folds            : {len(subjects)}")
    print(f"--> Max epochs per fold         : {epochs} (early stopping active)")
    print(f"--> Batch size                  : {batch_size}")
    print("="*60)

    for fold_idx, test_subject in enumerate(subjects):
        try:
            print(f"\n{'='*20} FOLD {fold_idx+1}/{len(subjects)}: Test → [{test_subject}] {'='*20}")

            # ── ON-THE-FLY DATA SPLITTING
            X_train_list, y_train_list = [], []
            X_test, y_test = None, None

            for file_path in all_files:
                curr_sub = os.path.basename(file_path).split('_')[0]
                X_sub, y_sub = load_subject_data(file_path, feature_keys)

                if curr_sub == test_subject:
                    X_test, y_test = X_sub, y_sub
                else:
                    X_train_list.append(X_sub)
                    y_train_list.append(y_sub)
                    # Free sub arrays reference — list holds the only reference now
                    del X_sub, y_sub

            X_train = np.concatenate(X_train_list, axis=0)
            y_train = np.concatenate(y_train_list, axis=0)

            # Free lists immediately after concatenation
            del X_train_list, y_train_list
            gc.collect()

            print(f"    Train shape : {X_train.shape} | Test shape : {X_test.shape}")

            # ── CHANNEL-WISE STANDARDISATION
            # Fit scaler on flattened train data, apply to test
            print("    Scaling features...")
            scaler = StandardScaler()
            X_train_flat = scaler.fit_transform(X_train.reshape(-1, num_channels))
            X_test_flat  = scaler.transform(X_test.reshape(-1, num_channels))

            X_train = X_train_flat.reshape(-1, num_timesteps, num_channels).astype(np.float32)
            X_test  = X_test_flat.reshape(-1, num_timesteps, num_channels).astype(np.float32)

            del X_train_flat, X_test_flat, scaler
            gc.collect()

            # ── BUILD FRESH LSTM MODEL FOR THIS FOLD
            model = build_lstm_model(
                input_shape=(num_timesteps, num_channels),
                num_classes=num_classes
            )

            # ── TRAIN
            history = model.fit(
                X_train, y_train,
                epochs=epochs,
                batch_size=batch_size,
                validation_split=0.1,
                callbacks=make_callbacks(),
                verbose=1
            )

            # ── EVALUATE
            y_pred_probs = model.predict(X_test, batch_size=batch_size)
            y_pred = np.argmax(y_pred_probs, axis=1)

            acc  = accuracy_score(y_test, y_pred)
            prec, rec, f1, _ = precision_recall_fscore_support(
                y_test, y_pred, average='macro', zero_division=0
            )

            print(f"\n    ✔ Fold [{test_subject}] → Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
            print("    Per-class breakdown:")
            print(classification_report(y_test, y_pred, zero_division=0))

            loso_results.append({
                'Subject'   : test_subject,
                'Accuracy'  : acc,
                'Precision' : prec,
                'Recall'    : rec,
                'F1_Score'  : f1,
                'Best_Epoch': len(history.history['loss'])
            })

            # ── AGGRESSIVE MEMORY PURGE after every fold
            del X_train, y_train, X_test, y_test, y_pred_probs, y_pred, model, history
            tf.keras.backend.clear_session()   # Clears TF graph + frees GPU allocations
            gc.collect()                        # Python GC pass

        except Exception as fold_err:
            raise RuntimeError(
                f"Pipeline crashed at fold [{test_subject}]\nDetails: {fold_err}"
            )

    # ── SAVE SUMMARY TABLE
    results_df = pd.DataFrame(loso_results)
    mean_row = results_df.mean(numeric_only=True).to_dict()
    std_row  = results_df.std(numeric_only=True).to_dict()
    mean_row['Subject'] = 'MEAN'
    std_row['Subject']  = 'STD_DEV'

    results_df = pd.concat(
        [results_df, pd.DataFrame([mean_row, std_row])],
        ignore_index=True
    )

    csv_path = os.path.join(output_dir, 'result', 'lstm_loso_metrics.csv')
    results_df.to_csv(csv_path, index=False)

    print("\n" + "="*22 + " FINAL LOSO REPORT " + "="*22)
    print(f"Mean Accuracy  : {mean_row['Accuracy']:.4f}  (± {std_row['Accuracy']:.4f})")
    print(f"Mean Precision : {mean_row['Precision']:.4f}  (± {std_row['Precision']:.4f})")
    print(f"Mean Recall    : {mean_row['Recall']:.4f}  (± {std_row['Recall']:.4f})")
    print(f"Mean F1-Score  : {mean_row['F1_Score']:.4f}  (± {std_row['F1_Score']:.4f})")
    print(f"Results saved  : {csv_path}")
    print("="*63)

    return results_df


print("[OK] LOSO function defined.")

## Cell 6 — Final Production Model (All Subjects)

In [ ]:
def train_final_lstm_model(data_dir, feature_keys, output_dir, epochs=15, batch_size=32):
    """
    Trains a final LSTM on ALL subjects for deployment/inference.
    Uses the same memory-efficient on-the-fly loading strategy.
    """
    print("\n" + "="*20 + " TRAINING FINAL PRODUCTION MODEL " + "="*20)

    all_files = sorted(glob.glob(os.path.join(data_dir, 'S*_windows_compressed.pkl.gz')))

    X_list, y_list = [], []
    for file_path in all_files:
        X_sub, y_sub = load_subject_data(file_path, feature_keys)
        X_list.append(X_sub)
        y_list.append(y_sub)
        del X_sub, y_sub  # Free immediately after append

    X_all = np.concatenate(X_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)
    del X_list, y_list
    gc.collect()

    _, num_timesteps, num_channels = X_all.shape
    num_classes = int(y_all.max() + 1)

    # Scale
    scaler = StandardScaler()
    X_all_flat = scaler.fit_transform(X_all.reshape(-1, num_channels))
    X_all = X_all_flat.reshape(-1, num_timesteps, num_channels).astype(np.float32)
    del X_all_flat
    gc.collect()

    # Build and train final model
    final_model = build_lstm_model(
        input_shape=(num_timesteps, num_channels),
        num_classes=num_classes
    )

    final_model.fit(
        X_all, y_all,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.05,   # Small hold-out to track loss during final training
        callbacks=make_callbacks(),
        verbose=1
    )

    # Save model
    model_path = os.path.join(output_dir, 'models', 'lstm_final_production.keras')
    final_model.save(model_path)
    print(f"\n[SAVED] Final LSTM model → {model_path}")

    del X_all, y_all, final_model
    tf.keras.backend.clear_session()
    gc.collect()


print("[OK] Final model training function defined.")

## Cell 7 — Run Full Pipeline

> ⚡ **Tip:** On Colab T4, each fold takes ~2–4 min. With 15 subjects + early stopping, expect ~40–60 min total.  
> Set `RUN_FINAL_MODEL = False` to skip the all-subjects training if you only want LOSO metrics.

In [ ]:
RUN_FINAL_MODEL = True   # Set to False to skip final production model training

try:
    # ── Phase 1: LOSO cross-validation
    print("\n" + "#"*60)
    print("  PHASE 1: LEAVE-ONE-SUBJECT-OUT CROSS-VALIDATION (LSTM)")
    print("#"*60)
    results_df = run_lstm_loso(
        data_dir     = COMPRESSED_DATA_DIR,
        feature_keys = SIGNAL_KEYS,
        output_dir   = OUTPUT_DIR,
        epochs       = 15,
        batch_size   = 32
    )

    # ── Phase 2: Final production model (optional)
    if RUN_FINAL_MODEL:
        print("\n" + "#"*60)
        print("  PHASE 2: FINAL PRODUCTION MODEL (ALL SUBJECTS)")
        print("#"*60)
        train_final_lstm_model(
            data_dir     = COMPRESSED_DATA_DIR,
            feature_keys = SIGNAL_KEYS,
            output_dir   = OUTPUT_DIR,
            epochs       = 15,
            batch_size   = 32
        )

    print("\n" + "✅"*10)
    print("  COMPLETE LSTM PIPELINE FINISHED SUCCESSFULLY")
    print("✅"*10)

except Exception as global_err:
    print("\n" + "#"*70)
    print("[PIPELINE TERMINATED — DETAILED ERROR BELOW]")
    print("#"*70)
    print(str(global_err))
    print("#"*70)

## Cell 8 — Results Visualisation
Run this after the pipeline completes to inspect per-subject performance.

In [ ]:
import matplotlib.pyplot as plt

# Load saved CSV (works even if you restart runtime and re-run just this cell)
csv_path   = os.path.join(OUTPUT_DIR, 'result', 'lstm_loso_metrics.csv')
results_df = pd.read_csv(csv_path)

# Exclude summary rows for plotting
plot_df = results_df[~results_df['Subject'].isin(['MEAN', 'STD_DEV'])].copy()
mean_row = results_df[results_df['Subject'] == 'MEAN'].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM — LOSO Cross-Validation Results (WESAD)', fontsize=14, fontweight='bold')

# ── Plot 1: Per-subject Accuracy and F1
x = range(len(plot_df))
axes[0].bar(x, plot_df['Accuracy'], alpha=0.7, label='Accuracy', color='steelblue', width=0.35)
axes[0].bar([i + 0.35 for i in x], plot_df['F1_Score'], alpha=0.7, label='F1-Score', color='coral', width=0.35)
axes[0].axhline(float(mean_row['Accuracy']), color='steelblue', linestyle='--', linewidth=1.2, label=f"Mean Acc={float(mean_row['Accuracy']):.3f}")
axes[0].axhline(float(mean_row['F1_Score']),  color='coral',    linestyle='--', linewidth=1.2, label=f"Mean F1={float(mean_row['F1_Score']):.3f}")
axes[0].set_xticks([i + 0.175 for i in x])
axes[0].set_xticklabels(plot_df['Subject'], rotation=45, ha='right', fontsize=8)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Score')
axes[0].set_title('Per-Subject Accuracy & F1-Score')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

# ── Plot 2: Precision, Recall, F1 comparison
metrics = ['Precision', 'Recall', 'F1_Score']
colours = ['#4C72B0', '#55A868', '#C44E52']
for i, (metric, colour) in enumerate(zip(metrics, colours)):
    axes[1].bar(
        [j + i * 0.25 for j in x],
        plot_df[metric], width=0.25,
        alpha=0.8, label=metric, color=colour
    )
axes[1].set_xticks([j + 0.25 for j in x])
axes[1].set_xticklabels(plot_df['Subject'], rotation=45, ha='right', fontsize=8)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Score')
axes[1].set_title('Per-Subject Precision / Recall / F1')
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'result', 'lstm_loso_plot.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"[SAVED] Plot → {plot_path}")

# ── Print final summary table
print("\n" + "="*55)
print("LSTM LOSO Summary Table")
print("="*55)
print(results_df.to_string(index=False))